In [2]:
import pandas as pd
from pathlib import Path

from iot_security_mlops.utils_data import df_sensor_msg_freq

# Determining Pre-training Tests
The purpose of this notebook is to analyze `../data/processed/reference.csv` and use the results as a basis for forming pre-training tests.

In [ ]:
path = Path('../data/processed/reference.csv')
specify_col_dtype = {'mqtt.clientid': str,
                     'mqtt.conack.flags': str,
                     'mqtt.conflags': str,
                     'mqtt.msg': str,
                     'mqtt.protoname': str}
df = pd.read_csv(path, dtype=specify_col_dtype)
df.head()

In [ ]:
time_delta = df.iloc[-1]['frame.time_epoch'] - df.iloc[0]['frame.time_epoch']
print(f'Total time elapsed in references.csv: {round(time_delta/60,2)} minutes')

## Check Null Proportions for each Column

In [5]:
df.isnull().sum() / len(df)

frame.time_delta              0.0
frame.time_delta_displayed    0.0
frame.time_epoch              0.0
frame.time_invalid            1.0
frame.time_relative           0.0
                             ... 
mqtt.willmsg_len              1.0
mqtt.willtopic                1.0
mqtt.willtopic_len            1.0
ip.proto                      0.0
class                         0.0
Length: 61, dtype: float64

All `frame.*` columns besides `frame.time_invalid` should not contain any nulls. Otherwise, the export from Wireshark is not valid and the rest of the data is suspect. Same case for `tcp.time_delta`. Wireshark will impute a value of 0.0 if it cannot find a previous packet in the same TCP stream. The aforementioned tests will be given a severity of "critical."

No nulls are recorded for `ip.src`, which is needed for measuring sensor health. So a no null constraint will be applied, with a severity of "critical". In a realistic IoT scenario, there could be nulls present due to packet corruption or packets that are not IP.

Not more than 40% of records should have null values for `mqtt.msgtype`. Otherwise, this could indicate that the sensors are no longer properly communicating with the MQTT broker. A severity of "warning" will be assigned since the network may still be functioning but network health must be checked.

## Pick Two Attributes to Test for their Distribution

### Motion Sensor (1)

In [6]:
df_sensor_msg_freq(df, '192.168.0.154', 3.0)['delta'].describe() #  in seconds

count    5072.000000
mean        1.000010
std         0.000777
min         0.998386
25%         0.999938
50%         1.000003
75%         1.000111
max         1.050016
Name: delta, dtype: float64

Apply a test for motion sensor (1) to check that the message frequency is between 0.99 and 1.06 seconds. Only assign a severity of 'warning' since frequencies outside this range does not mean that the sensor is broken. But it may be worth assessing their health.

### Motion Sensor (2)

In [7]:
df_sensor_msg_freq(df, '192.168.0.174', 3.0)['delta'].describe() #  in seconds

count    5063.000000
mean        1.001788
std         0.042131
min         0.998387
25%         0.999938
50%         1.000003
75%         1.000112
max         2.000185
Name: delta, dtype: float64

Apply a test for motion sensor (2) to check that the message frequency is between 0.99 and 2.01 seconds. Only assign a severity of 'warning' since frequencies outside this range does not mean that the sensor is broken. But it may be worth assessing their health.